In [ ]:
import inspect
from functools import update_wrapper
from typing import Callable, Generic, ParamSpec, TypeVar


P = ParamSpec("P")
R = TypeVar("R")


class JPath(Generic[P, R]):
    """
    A callable wrapper that preserves parameter and return type information.
    
    This class wraps a callable function while maintaining its original signature
    for static type checking and runtime introspection.
    
    Parameters
    ----------
    func : Callable[P, R]
        The function to wrap. Must have exactly three parameters:
        'env', 'partial_result', and 'path_options', in that order.
    
    Attributes
    ----------
    func : Callable[P, R]
        The wrapped function.
    __signature__ : inspect.Signature
        The signature of the original function, used by inspect.signature().
    
    Raises
    ------
    TypeError
        If the provided function does not have exactly the required parameters
        ('env', 'partial_result', 'path_options') in the correct order.
    
    Examples
    --------
    >>> @jmap
    ... def my_path(env: int, partial_result: str, path_options: bool) -> int:
    ...     return env
    
    >>> result = my_path(1, "test", True)
    >>> result
    1
    """
    
    REQUIRED_ARGS = ("env", "partial_result", "path_options")
    
    def __init__(self, func: Callable[P, R]) -> None:
        """
        Initialize the JPath wrapper.
        
        Parameters
        ----------
        func : Callable[P, R]
            The function to wrap.
        
        Raises
        ------
        TypeError
            If the function signature does not match requirements.
        """
        signature = inspect.signature(func)
        actual_args = tuple(signature.parameters.keys())
        
        if actual_args != self.REQUIRED_ARGS:
            raise TypeError(
                f"@jmap requires arguments {self.REQUIRED_ARGS}, in that order. "
                f"{func.__qualname__} has arguments {actual_args}."
            )
        
        self.func = func
        self.__signature__ = signature
        
        ################################# Analyze Schema ########################################
        self.env_trees, self.subpath_calls = analyze_env_schema(func)
    
    def __call__(self, *args: P.args, **kwargs: P.kwargs) -> R:
        """
        Call the wrapped function.
        
        Parameters
        ----------
        *args : P.args
            Positional arguments to pass to the wrapped function.
        **kwargs : P.kwargs
            Keyword arguments to pass to the wrapped function.
        
        Returns
        -------
        R
            The return value from the wrapped function.
        """
        print("In JPath")
        return self.func(*args, **kwargs)
    
    def migrate_version(
        self,
        update_fn: Callable,
        source_version: str | None = None,
        target_version: str | None = None,
    ) -> None:
        """
        Perform a migration of the JPath version.
        
        Parameters
        ----------
        update_fn : Callable
            A function that performs the version update operation.
        source_version : str, optional
            The source version to migrate from. Default is None.
        target_version : str, optional
            The target version to migrate to. Default is None.
        
        Returns
        -------
        None
        """
        pass
    
    def get_schema(self) -> set:
        """
        Extract and return the schema from all environment trees.
        
        Returns
        -------
        set
            A set containing unique usage entries from all environment trees.
        """
        # usage_list = []
        # for tree in self.env_trees:
        #     usage_list.extend(tree.usage_list())
        # return set(usage_list)
        pass


def jmap(func: Callable[P, R]) -> JPath[P, R]:
    """
    Decorator to convert a function into a JPath wrapper.
    
    This decorator preserves the original function's call signature for both
    static type checking and runtime introspection.
    
    Parameters
    ----------
    func : Callable[P, R]
        The function to wrap. Must have exactly three parameters:
        'env', 'partial_result', and 'path_options', in that order.
    
    Returns
    -------
    JPath[P, R]
        A JPath instance wrapping the original function.
    
    Examples
    --------
    >>> @jmap
    ... def my_path(env: int, partial_result: str, path_options: bool) -> int:
    ...     '''Test comments.'''
    ...     return env
    """
    return JPath(func)


@jmap
def my_path(
    env: int,
    partial_result: str,
    path_options: bool,
) -> int:
    """Test comments for MY PATH!!!"""
    print("Hello world")
    return env


# The IDE knows the original call signature.
result = my_path(1, "test", True)

# These should be flagged by a static type checker:
# my_path("wrong", "test", True)
# my_path(1, 123, True)
# my_path(1, "test")

# The IDE also knows that this is a JPath:
my_path.get_schema()

# And it knows JPath attributes:
original_function = my_path.func

# Runtime introspection also works.
print(inspect.signature(my_path))
# (env: int, partial_result: str, path_options: bool) -> int


In JPath
Hello world
(env: int, partial_result: str, path_options: bool) -> int
